# Práctica 11 — Búsqueda y recuperación clásica

**Curso:** NLP Clásico con Python · **Clase 11** · **Tiempo estimado:** 60-75 minutos
**Notebook de clase:** `notebooks_clase/clase11_busqueda_recuperacion_clasica.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio tiene una o varias celdas con `# TODO`: complétalas.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te indica qué revisar.
- Los 5 documentos de `DOCS_P` son diminutos y pensados para poder calcular a mano (indicado en cada ejercicio).
- Las **soluciones** están al final del notebook, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden. Librerías: `numpy`, `pandas`.

---
## Celda 0 — Setup y datos

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
print("Librerías listas.")

In [ ]:
import os
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.random.seed(42)                        # semilla fija (aunque este tema no entrena nada aleatorio)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

try:
    import nltk
    from nltk.corpus import stopwords
    stopwords.words("spanish")
except LookupError:
    nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
STOPWORDS = {"es": set(stopwords.words("spanish")), "it": set(stopwords.words("italian"))}   # dato-es

def tokenizar(texto, idioma, quitar_stopwords=True):
    """Minúsculas, solo letras (sin números ni puntuación), y stopwords opcionales."""
    palabras = re.findall(r"[^\W\d_]+", texto.lower())
    if quitar_stopwords:
        palabras = [p for p in palabras if p not in STOPWORDS[idioma]]
    return palabras

def construir_vocabulario(documentos_tok):
    """Lista ordenada de palabras distintas que aparecen en al menos un documento."""
    vocab = sorted({p for doc in documentos_tok for p in doc})
    return vocab, {p: i for i, p in enumerate(vocab)}

def matriz_tf(documentos_tok, indice):
    """Matriz de recuentos (documentos x vocabulario): TF crudo, sin normalizar."""
    M = np.zeros((len(documentos_tok), len(indice)), dtype=float)
    for i, doc in enumerate(documentos_tok):
        for palabra, n in Counter(doc).items():
            if palabra in indice:
                M[i, indice[palabra]] = n
    return M

def matriz_tfidf(TF, idf):
    """TF-IDF = TF * IDF, con cada fila normalizada a norma 1 (para que el coseno sea un producto escalar)."""
    X = TF * idf
    normas = np.linalg.norm(X, axis=1, keepdims=True)
    normas[normas == 0] = 1.0
    return X / normas

def dividir_frases(texto):
    """Corta un texto en frases por . ! ? y descarta las vacías."""
    return [f.strip() for f in re.split(r"(?<=[.!?])\s+", texto.strip()) if f.strip()]

print("Funciones auxiliares cargadas.")

In [ ]:
# datos-es
# Cinco documentos diminutos (idénticos en ES e IT: son datos) para los ejercicios 1 a 4
DOCS_P = ["la casa tiene un jardín grande", "el jardín de la casa tiene flores", "el coche está en el garaje",
          "el coche nuevo es rápido", "la bicicleta está en el jardín"]

In [ ]:
tok_p = [d.split() for d in DOCS_P]
vocab_p, idx_p = construir_vocabulario(tok_p)
TF_p = matriz_tf(tok_p, idx_p)
print("vocabulario:", vocab_p)
print("df de 'jardín':", int((TF_p[:, idx_p["jardín"]] > 0).sum()), "de 5 documentos")   # dato-es

## Ejercicio 1 — IDF del TF-IDF clásico

Completa `idf_tfidf_p(TF)`: `log((1+N)/(1+df)) + 1`, con `N` = número de documentos (filas de `TF`) y `df` el número de documentos en los que aparece cada palabra (columna con al menos un valor > 0). Con `DOCS_P` (5 documentos), `jardín` aparece en 3 de ellos: a mano, `idf = log(6/4) + 1 = 0,405 + 1 = 1,405`.

In [ ]:
def idf_tfidf_p(TF):
    return None    # TODO: log((1+N)/(1+df)) + 1, con N = TF.shape[0] y df = (TF > 0).sum(axis=0)

# Comprobación automática
_w_jardin, _w_garaje = "jardín", "garaje"   # dato-es
_idf = idf_tfidf_p(TF_p)
assert _idf is not None, "Completa las líneas marcadas con TODO"
assert np.isclose(_idf[idx_p[_w_jardin]], np.log(6 / 4) + 1), "a mano: log(6/4)+1 ≈ 1,405"
assert (_idf >= 1.0).all(), "el idf del TF-IDF clásico nunca baja de 1"
assert np.isclose(_idf[idx_p[_w_garaje]], np.log(6 / 2) + 1)   # aparece en 1 documento
print("Ejercicio 1 correcto ✔", round(float(_idf[idx_p[_w_jardin]]), 3))

## Ejercicio 2 — Vector TF-IDF de una consulta y búsqueda por coseno

Completa `buscar_coseno_p(consulta_tok, X, indice, idf, k)`: construye el vector TF-IDF de la consulta (TF de la consulta multiplicado por el `idf` de los documentos, sin olvidar normalizar a norma 1 si la norma no es 0) y devuelve los `k` documentos de `X` (ya normalizado por filas) con mayor producto escalar, junto con sus puntuaciones. Usa `Counter` para el TF de la consulta.

In [ ]:
def buscar_coseno_p(consulta_tok, X, indice, idf, k=3):
    v = np.zeros(len(indice))
    # TODO: rellena v[indice[palabra]] = recuento de la palabra en consulta_tok, para las palabras que estén en indice
    v = v * idf
    return None, None    # TODO: normaliza v (si su norma > 0), calcula X @ v, y devuelve los índices y puntuaciones de los k documentos con mayor puntuación (np.argsort)

# Comprobación automática
_idf_p = idf_tfidf_p(TF_p)
_X_p = matriz_tfidf(TF_p, _idf_p)
_orden, _punt = buscar_coseno_p(["coche", "garaje"], _X_p, idx_p, _idf_p, k=5)
assert _orden is not None, "Completa las líneas marcadas con TODO"
assert list(_orden[:2]) == [2, 3], "los documentos 2 y 3 (índice 0) hablan del coche/garaje; deberían ir primero"
assert _punt[0] > _punt[-1] and np.isclose(np.linalg.norm(_X_p[0]), 1.0)
_o2, _p2 = buscar_coseno_p(["palabra_inexistente"], _X_p, idx_p, _idf_p, k=5)
assert np.allclose(_p2, 0.0)   # consulta sin palabras del vocabulario: puntuación 0 en todos
print("Ejercicio 2 correcto ✔", list(_orden))

## Ejercicio 3 — IDF probabilístico de BM25 (y su corrección)

Completa `idf_bm25_p(TF, corregido=False)`: sin corregir, `log((N-df+0,5)/(df+0,5))`; con `corregido=True`, `log(1 + (N-df+0,5)/(df+0,5))`. Con `DOCS_P`, la palabra `el` aparece en 4 de los 5 documentos (más de la mitad): a mano, `idf_bm25(el) = log((5-4+0,5)/(4+0,5)) = log(1,5/4,5) = log(0,333) ≈ -1,099` (negativo).

In [ ]:
def idf_bm25_p(TF, corregido=False):
    N = TF.shape[0]
    df = (TF > 0).sum(axis=0)
    razon = None    # TODO: (N - df + 0.5) / (df + 0.5)
    return razon

# Comprobación automática
_w_el = "el"   # dato-es
_ib = idf_bm25_p(TF_p)
assert _ib is not None, "Completa las líneas marcadas con TODO"
assert np.isclose(_ib[idx_p[_w_el]], np.log(1.5 / 4.5)), "a mano: log(1,5/4,5) ≈ -1,099"
assert _ib[idx_p[_w_el]] < 0, "está en más de la mitad de los documentos: idf negativo"
_ib_c = idf_bm25_p(TF_p, corregido=True)
assert (_ib_c >= 0).all() and _ib_c[idx_p[_w_el]] > 0, "el idf corregido nunca es negativo"
print("Ejercicio 3 correcto ✔", "idf_bm25 =", round(float(_ib[idx_p[_w_el]]), 3), "| corregido =", round(float(_ib_c[idx_p[_w_el]]), 3))

## Ejercicio 4 — Precisión@k y MRR

Completa `precision_en_k_p(recuperados, relevantes, k)` (fracción de relevantes entre los `k` primeros de `recuperados`) y `rango_reciproco_p(recuperados, relevantes)` (`1/posición` del primer relevante, 1-indexada; `0.0` si ninguno de `recuperados` es relevante). A mano: `recuperados = [5, 2, 9, 1]`, `relevantes = {9, 1}`, `k=3` → entre `[5, 2, 9]` hay 1 relevante (`9`): `precisión@3 = 1/3`. El primer relevante (`9`) está en la posición 3: `rango_recíproco = 1/3`.

In [ ]:
def precision_en_k_p(recuperados, relevantes, k):
    top = recuperados[:k]
    return None    # TODO: número de elementos de top que están en relevantes, dividido por k

def rango_reciproco_p(recuperados, relevantes):
    pass    # TODO: recorre recuperados con enumerate(recuperados, 1); si el documento está en relevantes, devuelve 1/posición
    return 0.0

# Comprobación automática
_p3 = precision_en_k_p([5, 2, 9, 1], {9, 1}, 3)
assert _p3 is not None, "Completa las líneas marcadas con TODO"
assert np.isclose(_p3, 1 / 3), "a mano: 1 de 3 = 1/3"
assert np.isclose(precision_en_k_p([9, 1, 5, 2], {9, 1}, 2), 1.0)
assert np.isclose(precision_en_k_p([5, 2, 3], {9, 1}, 3), 0.0)
_rr = rango_reciproco_p([5, 2, 9, 1], {9, 1})
assert np.isclose(_rr, 1 / 3), "el primer relevante (9) está en la posición 3"
assert np.isclose(rango_reciproco_p([9, 5, 2], {9, 1}), 1.0)
assert np.isclose(rango_reciproco_p([5, 2, 3], {9, 1}), 0.0)
print("Ejercicio 4 correcto ✔", "precisión@3 =", round(_p3, 3), "| MRR de un ejemplo =", round(_rr, 3))

## Ejercicio 5 — Palabras clave con TF-IDF

Completa `palabras_clave_p(texto, idf_global, indice_global, n=3)`: tokeniza `texto` (usa `tokenizar(texto, "es")`), cuenta las palabras con `Counter`, puntúa cada una con `recuento * idf_global[indice_global[palabra]]` (solo las palabras que estén en `indice_global`) y devuelve las `n` palabras con mayor puntuación, de mayor a menor.

In [ ]:
def palabras_clave_p(texto, idf_global, indice_global, n=3):
    tok = tokenizar(texto, "es")   # dato-es
    cuenta = Counter(tok)
    puntuadas = []    # TODO: lista de (palabra, recuento * idf_global[indice_global[palabra]]) para las palabras de cuenta que estén en indice_global
    puntuadas.sort(key=lambda par: -par[1])
    return [p for p, _ in puntuadas[:n]]

# Comprobación automática
_idf_p3 = idf_tfidf_p(TF_p)
_texto_prueba = "el coche nuevo está en el garaje"   # dato-es
_claves = palabras_clave_p(_texto_prueba, _idf_p3, idx_p, n=3)
assert _claves, "Completa las líneas marcadas con TODO"
assert "garaje" in _claves, "'garaje' solo aparece en 1 de los 5 documentos: debería tener un idf alto y salir entre las claves"
_w_el2 = "el"   # dato-es
assert _w_el2 not in _claves, "aparece en más de la mitad de los documentos: idf bajo, no debería ser palabra clave"
print("Ejercicio 5 correcto ✔", _claves)

---
## Reto final — Resumen extractivo por puntuación de frase

Con `idf_tfidf_p`, `construir_vocabulario`, `matriz_tf` y `tokenizar` (ya disponibles) y `dividir_frases` (ya disponible):

1. Completa `resumen_extractivo_p(texto, idf_global, indice_global, n_frases=1)`: divide `texto` en frases (`dividir_frases`), puntúa cada frase con la **media** de `recuento_en_la_frase * idf_global[indice_global[palabra]]` sobre las palabras de la frase que estén en `indice_global` (si ninguna lo está, la puntuación es 0,0), y devuelve las `n_frases` con mayor puntuación **en su orden original** dentro del texto (no por puntuación).
2. Aplícalo al texto `TEXTO_RETO` (definido abajo, 3 frases) usando el IDF de `DOCS_P` y guarda el resultado en `resumen_reto`.
3. Comprueba que la frase elegida es la que menciona `jardín` y `flores`: las otras dos frases repiten `casa` y `coche`, palabras que también aparecen en más de un documento de `DOCS_P` y no bastan para ganar.

Tarda menos de un segundo.

In [ ]:
TEXTO_RETO = "El coche está en la casa. El jardín de la casa tiene flores. La casa tiene un coche."   # dato-es

def resumen_extractivo_p(texto, idf_global, indice_global, n_frases=1):
    frases = dividir_frases(texto)
    puntuaciones = []
    for frase in frases:
        pass    # TODO: tokeniza la frase con tokenizar (el idioma es el español), calcula la media de recuento*idf_global[...] sobre las palabras en indice_global (0.0 si ninguna lo está) y añádela a puntuaciones
    mejores = sorted(range(len(frases)), key=lambda i: -puntuaciones[i])[:n_frases]
    mejores.sort()
    return [frases[i] for i in mejores]

resumen_reto = None   # TODO: resumen_extractivo_p(TEXTO_RETO, idf_tfidf_p(TF_p), idx_p, n_frases=1)

# Comprobación automática
_w_jardin2, _w_flores = "jardín", "flores"   # dato-es
assert resumen_reto is not None, "Completa las líneas marcadas con TODO"
assert len(resumen_reto) == 1 and _w_jardin2 in resumen_reto[0].lower() and _w_flores in resumen_reto[0].lower(), "debería elegir la frase del jardín con flores"
print("Reto final correcto ✔", resumen_reto)

**Tus conclusiones (escribe 3-4 frases):** ¿por qué el IDF del TF-IDF clásico nunca es negativo, mientras que el de BM25 sí puede serlo? ¿Qué significa que la precisión@3 de una consulta con un único documento relevante no pueda superar 1/3? ¿Por qué una palabra clave necesita un IDF alto, no solo un TF alto? ¿Qué tienen en común la extracción de palabras clave y el resumen extractivo?

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
def idf_tfidf_p(TF):
    N = TF.shape[0]
    df = (TF > 0).sum(axis=0)
    return np.log((1 + N) / (1 + df)) + 1.0

# Comprobación automática
_w_jardin, _w_garaje = "jardín", "garaje"   # dato-es
_idf = idf_tfidf_p(TF_p)
assert _idf is not None, "Completa las líneas marcadas con TODO"
assert np.isclose(_idf[idx_p[_w_jardin]], np.log(6 / 4) + 1), "a mano: log(6/4)+1 ≈ 1,405"
assert (_idf >= 1.0).all(), "el idf del TF-IDF clásico nunca baja de 1"
assert np.isclose(_idf[idx_p[_w_garaje]], np.log(6 / 2) + 1)   # aparece en 1 documento
print("Ejercicio 1 correcto ✔", round(float(_idf[idx_p[_w_jardin]]), 3))

### Solución del ejercicio 2

In [ ]:
def buscar_coseno_p(consulta_tok, X, indice, idf, k=3):
    v = np.zeros(len(indice))
    for palabra, n in Counter(consulta_tok).items():
        if palabra in indice:
            v[indice[palabra]] = n
    v = v * idf
    norma = np.linalg.norm(v)
    if norma > 0:
        v = v / norma
    puntuaciones = X @ v
    orden = np.argsort(-puntuaciones)[:k]
    return orden, puntuaciones[orden]

# Comprobación automática
_idf_p = idf_tfidf_p(TF_p)
_X_p = matriz_tfidf(TF_p, _idf_p)
_orden, _punt = buscar_coseno_p(["coche", "garaje"], _X_p, idx_p, _idf_p, k=5)
assert _orden is not None, "Completa las líneas marcadas con TODO"
assert list(_orden[:2]) == [2, 3], "los documentos 2 y 3 (índice 0) hablan del coche/garaje; deberían ir primero"
assert _punt[0] > _punt[-1] and np.isclose(np.linalg.norm(_X_p[0]), 1.0)
_o2, _p2 = buscar_coseno_p(["palabra_inexistente"], _X_p, idx_p, _idf_p, k=5)
assert np.allclose(_p2, 0.0)   # consulta sin palabras del vocabulario: puntuación 0 en todos
print("Ejercicio 2 correcto ✔", list(_orden))

### Solución del ejercicio 3

In [ ]:
def idf_bm25_p(TF, corregido=False):
    N = TF.shape[0]
    df = (TF > 0).sum(axis=0)
    razon = (N - df + 0.5) / (df + 0.5)
    return np.log(1.0 + razon) if corregido else np.log(razon)

# Comprobación automática
_w_el = "el"   # dato-es
_ib = idf_bm25_p(TF_p)
assert _ib is not None, "Completa las líneas marcadas con TODO"
assert np.isclose(_ib[idx_p[_w_el]], np.log(1.5 / 4.5)), "a mano: log(1,5/4,5) ≈ -1,099"
assert _ib[idx_p[_w_el]] < 0, "está en más de la mitad de los documentos: idf negativo"
_ib_c = idf_bm25_p(TF_p, corregido=True)
assert (_ib_c >= 0).all() and _ib_c[idx_p[_w_el]] > 0, "el idf corregido nunca es negativo"
print("Ejercicio 3 correcto ✔", "idf_bm25 =", round(float(_ib[idx_p[_w_el]]), 3), "| corregido =", round(float(_ib_c[idx_p[_w_el]]), 3))

### Solución del ejercicio 4

In [ ]:
def precision_en_k_p(recuperados, relevantes, k):
    top = recuperados[:k]
    return len([d for d in top if d in relevantes]) / k

def rango_reciproco_p(recuperados, relevantes):
    for i, d in enumerate(recuperados, 1):
        if d in relevantes:
            return 1.0 / i
    return 0.0

# Comprobación automática
_p3 = precision_en_k_p([5, 2, 9, 1], {9, 1}, 3)
assert _p3 is not None, "Completa las líneas marcadas con TODO"
assert np.isclose(_p3, 1 / 3), "a mano: 1 de 3 = 1/3"
assert np.isclose(precision_en_k_p([9, 1, 5, 2], {9, 1}, 2), 1.0)
assert np.isclose(precision_en_k_p([5, 2, 3], {9, 1}, 3), 0.0)
_rr = rango_reciproco_p([5, 2, 9, 1], {9, 1})
assert np.isclose(_rr, 1 / 3), "el primer relevante (9) está en la posición 3"
assert np.isclose(rango_reciproco_p([9, 5, 2], {9, 1}), 1.0)
assert np.isclose(rango_reciproco_p([5, 2, 3], {9, 1}), 0.0)
print("Ejercicio 4 correcto ✔", "precisión@3 =", round(_p3, 3), "| MRR de un ejemplo =", round(_rr, 3))

### Solución del ejercicio 5

In [ ]:
def palabras_clave_p(texto, idf_global, indice_global, n=3):
    tok = tokenizar(texto, "es")   # dato-es
    cuenta = Counter(tok)
    puntuadas = [(p, n_ * idf_global[indice_global[p]]) for p, n_ in cuenta.items() if p in indice_global]
    puntuadas.sort(key=lambda par: -par[1])
    return [p for p, _ in puntuadas[:n]]

# Comprobación automática
_idf_p3 = idf_tfidf_p(TF_p)
_texto_prueba = "el coche nuevo está en el garaje"   # dato-es
_claves = palabras_clave_p(_texto_prueba, _idf_p3, idx_p, n=3)
assert _claves, "Completa las líneas marcadas con TODO"
assert "garaje" in _claves, "'garaje' solo aparece en 1 de los 5 documentos: debería tener un idf alto y salir entre las claves"
_w_el2 = "el"   # dato-es
assert _w_el2 not in _claves, "aparece en más de la mitad de los documentos: idf bajo, no debería ser palabra clave"
print("Ejercicio 5 correcto ✔", _claves)

### Solución del reto final

In [ ]:
TEXTO_RETO = "El coche está en la casa. El jardín de la casa tiene flores. La casa tiene un coche."   # dato-es

def resumen_extractivo_p(texto, idf_global, indice_global, n_frases=1):
    frases = dividir_frases(texto)
    puntuaciones = []
    for frase in frases:
        tok = tokenizar(frase, "es")   # dato-es
        cuenta = Counter(tok)
        pesos = [cuenta[p] * idf_global[indice_global[p]] for p in cuenta if p in indice_global]
        puntuaciones.append(float(np.mean(pesos)) if pesos else 0.0)
    mejores = sorted(range(len(frases)), key=lambda i: -puntuaciones[i])[:n_frases]
    mejores.sort()
    return [frases[i] for i in mejores]

resumen_reto = resumen_extractivo_p(TEXTO_RETO, idf_tfidf_p(TF_p), idx_p, n_frases=1)

# Comprobación automática
_w_jardin2, _w_flores = "jardín", "flores"   # dato-es
assert resumen_reto is not None, "Completa las líneas marcadas con TODO"
assert len(resumen_reto) == 1 and _w_jardin2 in resumen_reto[0].lower() and _w_flores in resumen_reto[0].lower(), "debería elegir la frase del jardín con flores"
print("Reto final correcto ✔", resumen_reto)

---
## Autoevaluación de la clase

Responde con honestidad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto no te ha quedado claro (TF-IDF frente a BM25, el IDF negativo, precisión@k frente a MRR, palabras clave, resumen extractivo)?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*